# Comparar fonética: atual × Luis

Compara o tratamento em [`features.py`](../features.py) (`clean_name` +
`full_name_phon_basic`, o mesmo caminho do pipeline) com
[`fonetica_luis.py`](../fonetica_luis.py).

**Já alinhado com produção:** Ç/Ñ, TH/RH, QUE/QUI, GUI/GUE, NH/LH, partículas/placeholders.

**Só no Luis (falado):** S entre vogais → Z, finais O→U / E→I / L→U / M→N, nasal M+cons→N, epêntese `^S`+cons.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from features import clean_name, full_name_phon_basic
from fonetica_luis import fonetica, limpar

def fonetica_atual(nome: str) -> str:
    """Caminho de produção: limpa partículas/placeholders e aplica fonética."""
    limpo = clean_name(nome)
    if not limpo:
        return ''
    return full_name_phon_basic(limpo)

SAMPLE = [
    'MARIA DA SILVA',
    'JOSE',
    'JOSÉ',
    'JOÃO',
    'PHILIPPE',
    'SCHMIDT',
    'SHIRLEY',
    'CHAVES',
    'GUIMARAES',
    'GUIMARÃES',
    'QUEIROZ',
    'ASSUNÇÃO',
    'ASSUNCAO',
    'CONCEIÇÃO',
    'WAGNER',
    'WILLIAM',
    'THEODORO',
    'RHUAN',
    'CKRISTIAN',
    'ANNA',
    'SMITH',
    'ABDALA',
    'ÑANDU',
    'MUZAMBINHO',
    'CAMPOS',
    'SAMPAIO',
    'MARIA-CLARA',
    'DESCONHECIDO',
    'MAE',
]

# Regressão leve do Luis (não deve voltar FILIPIPI / QUE sem U).
assert fonetica('PHILIPPE') in {'FILIPE', 'FILIPI'}
assert fonetica('QUEIROZ').startswith('KEI')
assert 'GI' in fonetica('GUIMARÃES') or fonetica('GUIMARÃES').startswith('JI')
assert fonetica_atual('ASSUNÇÃO') == 'ASUNSAO'
print('atual =', fonetica_atual('PHILIPPE SCHMIDT'))
print('luis  =', fonetica('PHILIPPE SCHMIDT'))


## Sample fixo

`igual` = as duas fonéticas coincidem. Foque nas linhas `False`.


In [ ]:
rows = []
for nome in SAMPLE:
    a = fonetica_atual(nome)
    b = fonetica(nome)
    rows.append({
        'nome': nome,
        'limpo_atual': clean_name(nome) or '',
        'limpo_luis': limpar(nome),
        'phon_atual': a,
        'phon_luis': b,
        'igual': a == b,
    })

df = pd.DataFrame(rows)
print(f"Iguais: {df['igual'].sum()} / {len(df)} | Diferentes: {(~df['igual']).sum()}")
display(df)
display(df.loc[~df['igual'], ['nome', 'phon_atual', 'phon_luis']])


## Token a token (só onde diverge)

Útil para ver se a diferença é limpeza (partículas) ou a regra fonética.


In [ ]:
def tokens(s: str) -> list[str]:
    return s.split() if s else []

div = df.loc[~df['igual']].copy()
detalhe = []
for r in div.itertuples(index=False):
    ta, tb = tokens(r.phon_atual), tokens(r.phon_luis)
    n = max(len(ta), len(tb))
    for i in range(n):
        detalhe.append({
            'nome': r.nome,
            'i': i,
            'tok_atual': ta[i] if i < len(ta) else None,
            'tok_luis': tb[i] if i < len(tb) else None,
            'tok_igual': (ta[i] if i < len(ta) else None) == (tb[i] if i < len(tb) else None),
        })
display(pd.DataFrame(detalhe))


## Opcional: amostra da base limpa

Se `censo_limpo` / `cpf_limpo` existirem, pega nomes distintos e mede a taxa de
divergência. Sem DuckDB/artefatos, a célula só avisa.


In [ ]:
N_SAMPLE_BASE = 500

try:
    from config import (
        TABELA_CENSO_LIMPA,
        TABELA_CPF_LIMPA,
        get_connection,
        list_tables,
        require_tables,
    )
    con = get_connection()
    tabelas = list_tables(con)
    if TABELA_CENSO_LIMPA in tabelas and TABELA_CPF_LIMPA in tabelas:
        nomes = con.execute(f'''
        SELECT nome_completo
        FROM (
            SELECT DISTINCT nome_completo FROM {TABELA_CENSO_LIMPA}
            WHERE nome_completo IS NOT NULL
            UNION
            SELECT DISTINCT nome_completo FROM {TABELA_CPF_LIMPA}
            WHERE nome_completo IS NOT NULL
        )
        USING SAMPLE {N_SAMPLE_BASE} ROWS
        ''').df()['nome_completo'].tolist()
        rows_b = []
        for nome in nomes:
            a = fonetica_atual(nome)
            b = fonetica(nome)
            if a != b:
                rows_b.append({'nome': nome, 'phon_atual': a, 'phon_luis': b})
        print(
            f'Amostra base: {len(nomes):,} | divergentes: {len(rows_b):,} '
            f'({100.0 * len(rows_b) / len(nomes) if nomes else 0:.1f}%)'
        )
        display(pd.DataFrame(rows_b).head(40))
    else:
        print('Bases limpas ausentes — rode o NB00b ou use só o sample fixo.')
    con.close()
except Exception as e:
    print('Pulando amostra da base:', type(e).__name__, e)
